# Accruals Quality — Validation Notebook

**Goal:** Compute accruals from v2 DB, compare against v1 output.  
**Components:** CF accruals ratio, BS accruals ratio, EPS CV, beat rate → composite signal  
**Pass criteria:** Correlation ≥ 0.90 on each component. Composite within 0.10 for ≥90% of stocks.

In [ ]:
import sys, os
os.chdir(os.path.expanduser("~/alpha-signal-v2"))
sys.path.insert(0, ".")

import pandas as pd
import numpy as np
from signals.accruals import _load_data, _compute_scores

## Step 1: Compute v2 scores

In [ ]:
stocks, qi, bs, cf = _load_data()
v2 = _compute_scores(stocks, qi, bs, cf)
print(f"v2: {len(v2)} stocks")
for col in ["cf_accruals_ratio", "bs_accruals_ratio", "earnings_persistence", "accruals_signal"]:
    n = v2[col].notna().sum()
    print(f"  {col}: {n} non-null")

## Step 2: Merge with v1 and compare components

In [ ]:
v1 = pd.read_csv(os.path.expanduser("~/alpha-signal/data/signals/accruals.csv"))
print(f"v1: {len(v1)} stocks")

merged = v2.merge(v1[["sid", "cf_accruals_ratio", "bs_accruals_ratio", "eps_cv",
                       "earnings_beat_rate", "accruals_signal"]],
                  on="sid", how="inner", suffixes=("_v2", "_v1"))
print(f"Overlapping: {len(merged)}")

# Compare each component
pairs = [
    ("cf_accruals_ratio_v2", "cf_accruals_ratio_v1", "CF Accruals"),
    ("bs_accruals_ratio_v2", "bs_accruals_ratio_v1", "BS Accruals"),
    ("earnings_persistence", "eps_cv", "EPS CV"),
    ("accruals_signal_v2", "accruals_signal_v1", "Composite Signal"),
]

print(f"\n{'Component':<20} {'Both valid':>10} {'Corr':>8} {'Mean diff':>10}")
print("-" * 52)
for v2c, v1c, label in pairs:
    both = merged[[v2c, v1c]].dropna()
    n = len(both)
    if n > 10:
        corr = both[v2c].corr(both[v1c])
        diff = (both[v2c] - both[v1c]).abs().mean()
        print(f"{label:<20} {n:>10} {corr:>8.4f} {diff:>10.4f}")
    else:
        print(f"{label:<20} {n:>10}   insufficient data")

## Step 3: Composite signal agreement

In [ ]:
both_sig = merged[["accruals_signal_v2", "accruals_signal_v1"]].dropna()
both_sig["diff"] = (both_sig["accruals_signal_v2"] - both_sig["accruals_signal_v1"]).abs()

within_010 = (both_sig["diff"] <= 0.10).sum()
within_020 = (both_sig["diff"] <= 0.20).sum()
pct_010 = within_010 / len(both_sig) * 100
pct_020 = within_020 / len(both_sig) * 100

print(f"Composite signal comparison ({len(both_sig)} stocks):")
print(f"  Within ±0.10: {within_010}/{len(both_sig)} = {pct_010:.1f}%")
print(f"  Within ±0.20: {within_020}/{len(both_sig)} = {pct_020:.1f}%")
print(f"  Correlation:   {both_sig['accruals_signal_v2'].corr(both_sig['accruals_signal_v1']):.4f}")
print(f"  Mean |diff|:   {both_sig['diff'].mean():.4f}")
print(f"\n  v2 mean={both_sig['accruals_signal_v2'].mean():.3f}  v1 mean={both_sig['accruals_signal_v1'].mean():.3f}")

## Step 4: Outlier investigation

In [ ]:
outliers = both_sig[both_sig["diff"] > 0.20].sort_values("diff", ascending=False)
if len(outliers) > 0:
    print(f"Stocks with |diff| > 0.20: {len(outliers)}")
    print(outliers.head(10).to_string())
else:
    print("No outliers with |diff| > 0.20")

## Step 5: Save to DB (only if validation passed)

In [ ]:
# from signals.accruals import compute
# compute(dry_run=False)